# Knowledge Retrieval (RAG)

Retrieval-Augmented Generation (RAG) grounds LLM responses in external, up-to-date knowledge: retrieve relevant document chunks → augment the prompt → generate a grounded answer.

## Implementation with Flyte v2 + the Agent harness

The original LangChain + LangGraph version wired a fixed `StateGraph`: **always** retrieve, then **always** generate. This notebook refactors it into a Flyte v2 `Agent` with a single `retrieve_documents` tool (ChromaDB + sentence-transformers, in-memory, zero infrastructure). Retrieval becomes **agentic**: the model decides _when_ it needs to look something up versus answering directly — and can retrieve more than once if the first pass is insufficient.

#### LangChain / LangGraph vs Flyte v2 + Agent harness

| Aspect | LangChain / LangGraph | Flyte v2 + `Agent` harness |
|--------|----------------------|----------------------------|
| **Pipeline shape** | Fixed `retrieve → generate` edges | Agent decides when to call `retrieve_documents` |
| **Re-retrieval** | Needs an explicit cycle node | Agent can retrieve again within `max_turns` |
| **Vector store** | Weaviate (external service) | ChromaDB in-memory (zero infrastructure) |
| **Retrieval step** | A graph node | A traced `@env.task` tool, cached with `cache="auto"` |
| **LLM client** | `ChatOpenAI` | Harness' litellm callback |
| **Output** | `TypedDict` state | Typed `AgentResult` |
| **Secrets** | `.env` / `os.environ` | `flyte.Secret` injected by cluster |

### 1. Install dependencies

In [ ]:
!uv pip install 'flyte[tui]' litellm chromadb sentence-transformers

### Start the devbox

If you haven't already, install the flyte package with the command above, then launch the local cluster:

In [ ]:
!flyte start devbox

### 2. Store your API key

In [ ]:
!flyte create secret ANTHROPIC_API_KEY --value sk-proj-...

### 3. Import dependencies and configure the TaskEnvironment

In [ ]:
from __future__ import annotations

import os
from datetime import timedelta

import flyte
from flyte.ai.agents import Agent, AgentResult

flyte.init_from_config()

_image = (
    flyte.Image.from_debian_base(name="rag-agent", python_version=(3, 12))
    .with_pip_packages("litellm", "chromadb>=0.5.0", "sentence-transformers>=2.0.0")
)

rag_env = flyte.TaskEnvironment(
    name="rag_pipeline",
    image=_image,
    resources=flyte.Resources(cpu="2", memory="4Gi"),
    secrets=[
        flyte.Secret(key="ANTHROPIC_API_KEY", as_env_var="ANTHROPIC_API_KEY"),
    ],
)

### 4. Define the retrieval tool

The LangGraph `StateGraph` had two nodes wired by a fixed edge: `retrieve_documents_node` → `generate_response_node`. In the Agent harness, retrieval becomes **one tool the model calls on demand**; generation is the agent's own job, so there is no separate `generate_task`.

`retrieve_documents` is a normal `@env.task` (it runs in a container and appears as a nested, traced sub-action) wrapping ChromaDB + sentence-transformers. `cache="auto"` means a repeated `(question, top_k)` skips re-embedding. The corpus lives next to the tool as a module constant so it ships with the task.

In [ ]:
KNOWLEDGE_BASE = """
Flyte is a cloud-native workflow orchestration platform designed for machine learning and data pipelines.
It was originally developed at Lyft and open-sourced in 2021. Flyte provides a type-safe, reproducible,
and scalable way to define and execute workflows using Python. Tasks in Flyte are containerized functions
that run on Kubernetes. Each task can be assigned specific compute resources like CPU, memory, and GPU.
Flyte supports caching of task outputs, which means if a task has already run with the same inputs, it
returns the cached result instead of recomputing. The Flyte v2 SDK uses a TaskEnvironment to group tasks
that share the same container image and resource configuration. Flyte also supports reusable containers
through ReusePolicy, which keeps warm containers around to reduce cold-start latency. Secrets are managed
securely through the flyte.Secret API, which injects credentials as environment variables at task execution
time without storing them in code. The Flyte UI provides real-time visibility into workflow execution,
including task inputs, outputs, logs, and custom HTML reports.
"""


def _chunk_text(text: str, chunk_size: int = 500, overlap: int = 50) -> list[str]:
    """Split text into overlapping chunks."""
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        boundary = text.rfind(".", start, end)
        if boundary > start + chunk_size // 2:
            end = boundary + 1
        chunks.append(text[start:end].strip())
        start = end - overlap
    return [c for c in chunks if c]


@rag_env.task(cache="auto")
async def retrieve_documents(question: str, top_k: int = 3) -> str:
    """Search the knowledge base for passages relevant to a question.

    Call this whenever the user asks about Flyte and you need specific facts to
    answer accurately.

    Args:
        question: The information need to search for.
        top_k: How many passages to return.
    """
    import chromadb
    from sentence_transformers import SentenceTransformer

    model = SentenceTransformer("all-MiniLM-L6-v2")
    chunks = _chunk_text(KNOWLEDGE_BASE)
    embeddings = model.encode(chunks).tolist()
    query_embedding = model.encode([question])[0].tolist()

    client = chromadb.Client()
    # Unique collection name per call avoids a name clash on a warm/reused container.
    name = f"rag_{abs(hash(question)) % 10_000_000}"
    collection = client.create_collection(name)
    collection.add(
        embeddings=embeddings,
        documents=chunks,
        ids=[str(i) for i in range(len(chunks))],
    )
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=min(top_k, len(chunks)),
    )
    client.delete_collection(name)

    passages = results["documents"][0]
    return "\n\n".join(f"[passage {i}] {p}" for i, p in enumerate(passages))

### 5. Build the RAG agent

#### From fixed pipeline to agentic retrieval

The plain-task version hard-wired `retrieve → generate`: every question paid for an embedding lookup, even "hello". The agent instead **decides** whether to retrieve. It also closes the loop the `StateGraph` couldn't without an explicit cycle: if the first passages don't answer the question, the model can call `retrieve_documents` again before answering.

In [ ]:
rag_agent = Agent(
    name="rag-assistant",
    model="claude-haiku-4-5",
    instructions=(
        "You answer questions about Flyte. When a question needs specific facts, call "
        "retrieve_documents to fetch relevant passages, then answer ONLY from what you "
        "retrieved and keep it to about 3 sentences. For greetings or general chit-chat, "
        "answer directly without retrieving. If the passages don't contain the answer, "
        "say you don't know."
    ),
    tools=[retrieve_documents],
    max_turns=4,
)


@rag_env.task(cache=flyte.Cache(behavior="disable"))
async def rag_pipeline(question: str) -> str:
    """Agentic RAG: the agent decides whether to retrieve before answering."""
    result: AgentResult = await rag_agent.run.aio(question)
    if result.error:
        raise RuntimeError(result.error)
    return result.summary

### 7. Run locally

In [ ]:
QUESTIONS = [
    "What is Flyte and where was it originally developed?",
    "How does Flyte handle secrets?",
    "What is a TaskEnvironment in Flyte v2?",
]

for question in QUESTIONS:
    run = flyte.run(rag_pipeline, question=question)
    run.wait()
    answer = run.outputs()[0]
    print(f"Q: {question}")
    print(f"A: {answer}")
    print()

### Running remotely

`retrieve_documents` carries `cache="auto"`, so repeated queries against the same knowledge base skip re-embedding — reducing latency and cost on subsequent runs. Because the tool is an `@env.task`, every retrieval also appears as its own nested action in the Flyte UI, right under the parent `agent.run`.

In [ ]:
run = flyte.run(
    rag_pipeline,
    question="How does caching work in Flyte?",
)
run.wait()
print(run.outputs()[0])